# 04 - Watch Errors Cascade Across Steps

Evaluates the first composed agentic behavior: route selection followed by retrieval. It compares retrieval using the expected source versus the router-predicted source and shows how routing errors cascade into retrieval failures. Supports both base and challenging datasets.


## Learning Goal

Connect two component evals: routing and retrieval. This lab shows students how a correct-looking retrieval system can still fail when an upstream routing decision sends the query to the wrong source.

## Where This Fits

Progression: data sanity -> router eval -> retrieval eval -> cascade eval -> answer quality -> full benchmark -> ablation.

This is the first composed behavior eval. It asks: when the router predicts a source, what happens to retrieval quality downstream?

## Related AI Evals Concepts

- Traces For Evals: inspect upstream decisions before diagnosing downstream failures.
- Transition Matrix: route -> retrieve is a simple transition where failures can propagate.
- Error Analysis: compare expected-source retrieval against predicted-source retrieval to find failure patterns.
- Don't Use Generic Eval Metrics: measure route-induced retrieval failures directly.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

PROJECT_ROOT


In [ ]:
import asyncio
import importlib
import os
from typing import Any

import chromadb
import pandas as pd

import agentic_rag.ingestion as ingestion_module  # noqa: E402
import agentic_rag.retrievers as retrievers_module  # noqa: E402
import agentic_rag.router as router_module  # noqa: E402
from agentic_rag.constants import SourceType  # noqa: E402
from agentic_rag.evaluation import parse_doc_ids, score_retrieval  # noqa: E402

importlib.reload(ingestion_module)
importlib.reload(retrievers_module)
importlib.reload(router_module)

from agentic_rag.ingestion import ensure_chroma_collections  # noqa: E402
from agentic_rag.llm import OpenAITextGenerator  # noqa: E402
from agentic_rag.retrievers import ChromaRetriever  # noqa: E402
from agentic_rag.router import QueryRouter  # noqa: E402
from agentic_rag.settings import Settings  # noqa: E402
from agentic_rag.telemetry import configure_tracing, start_span  # noqa: E402

pd.set_option("display.max_colwidth", 180)


In [ ]:
TRACE_DIR = PROJECT_ROOT / "otel_traces"
TRACE_DIR.mkdir(exist_ok=True)
TRACE_FILE = TRACE_DIR / "04_router_retrieval_cascade.jsonl"

trace_settings = Settings(
    _env_file=None,
    OTEL_TRACING_ENABLED=True,
    OTEL_TRACES_EXPORTER="file",
    OTEL_TRACES_FILE=TRACE_FILE,
    OTEL_SERVICE_NAME="agentic-rag-notebooks",
)
configure_tracing(trace_settings)

TRACE_FILE


## Load Router And Retrieval Test Sets


In [ ]:
base_df = pd.read_csv(PROJECT_ROOT / "datasets/evaluation_dataset.csv")
challenge_df = pd.read_csv(PROJECT_ROOT / "datasets/challenging_router_evaluation_dataset.csv")

if "Query" in base_df.columns:
    base_df = base_df.rename(columns={"Query": "query", "Expected_Source_Type": "expected_source_type"})

base_df["dataset"] = "base"
challenge_df["dataset"] = "challenging"
base_df["expected_source_type"] = base_df["expected_source_type"].astype(str)
challenge_df["expected_source_type"] = challenge_df["expected_source_type"].astype(str)

datasets = pd.concat([base_df, challenge_df], ignore_index=True, sort=False)
datasets[["dataset", "query", "expected_source_type"]].head()


In [ ]:
qna_df = pd.read_csv(PROJECT_ROOT / "datasets/medical_qna_dataset.csv")
device_df = pd.read_csv(PROJECT_ROOT / "datasets/medical_device_manuals_dataset.csv")

settings = Settings(_env_file=None, chroma_path=PROJECT_ROOT / "chroma_db")
client = chromadb.PersistentClient(path=str(settings.chroma_path))

collection_summary = ensure_chroma_collections(client, qna_df, device_df)
collection_summary


## Define Cascade Metrics


The CSV `expected_doc_ids` values are local eval-file row numbers, while Chroma returns IDs such as `qna-7354` and `device-346`. Cascade retrieval metrics compare IDs by exact string match, so this notebook resolves the gold rows to Chroma document IDs before scoring and keeps the original CSV value in `csv_expected_doc_ids`.


In [ ]:
LABELS = [source.value for source in SourceType]
LOCAL_SOURCES = {SourceType.RETRIEVE_QNA.value, SourceType.RETRIEVE_DEVICE.value}
settings = Settings(_env_file=None, chroma_path=PROJECT_ROOT / "chroma_db")


def normalize_text(value: object) -> str:
    return " ".join(str(value or "").lower().split())


qna_id_by_question_answer = {
    (normalize_text(row["Question"]), normalize_text(row["Answer"])): f"qna-{row_index}"
    for row_index, row in qna_df.reset_index(drop=True).iterrows()
}
qna_ids_by_question = qna_df.reset_index(drop=True).groupby(
    qna_df["Question"].map(normalize_text)
).apply(lambda rows: [f"qna-{row_index}" for row_index in rows.index]).to_dict()

device_answer_columns = ["Indications_for_Use", "Contraindications", "Patient_Population"]
device_lookup_rows = []
for row_index, row in device_df.reset_index(drop=True).iterrows():
    for column in device_answer_columns:
        if column in row and not pd.isna(row[column]):
            device_lookup_rows.append(
                {
                    "doc_id": f"device-{row_index}",
                    "answer": normalize_text(row[column]),
                    "device_name": normalize_text(row["Device_Name"]),
                    "model_number": normalize_text(row["Model_Number"]),
                }
            )


def source_or_none(value: str) -> SourceType | None:
    try:
        return SourceType(value)
    except ValueError:
        return None


def is_local_source(value: str) -> bool:
    return value in LOCAL_SOURCES


def resolve_expected_doc_ids(row: pd.Series) -> list[str]:
    source = source_or_none(str(row["expected_source_type"]))
    query = normalize_text(row["query"])
    expected_answer = normalize_text(row.get("expected_answer", ""))

    if source == SourceType.RETRIEVE_QNA:
        exact_match = qna_id_by_question_answer.get((query, expected_answer))
        if exact_match:
            return [exact_match]
        return qna_ids_by_question.get(query, [])

    if source == SourceType.RETRIEVE_DEVICE:
        candidates = [item for item in device_lookup_rows if item["answer"] == expected_answer]
        model_matches = [item for item in candidates if item["model_number"] and item["model_number"] in query]
        if model_matches:
            return sorted({item["doc_id"] for item in model_matches})
        named_matches = [item for item in candidates if item["device_name"] and item["device_name"] in query]
        if named_matches:
            return sorted({item["doc_id"] for item in named_matches})
        return sorted({item["doc_id"] for item in candidates})

    return []


def expected_ids_from_row(row: pd.Series) -> list[str]:
    resolved_ids = resolve_expected_doc_ids(row)
    return resolved_ids or parse_doc_ids(row.get("expected_doc_ids", ""))


def lexical_score(question: str, answer: str) -> float:
    q_tokens = {token.lower().strip(".,?!:;()[]{}\"'") for token in question.split() if len(token) > 3}
    a_tokens = {token.lower().strip(".,?!:;()[]{}\"'") for token in answer.split() if len(token) > 3}
    if not q_tokens or not a_tokens:
        return 0.0
    return len(q_tokens & a_tokens) / len(q_tokens)


def preview_text(text: str, max_words: int = 50) -> str:
    return " ".join(text.split()[:max_words])

async def route_and_retrieve_one(row: pd.Series, top_k: int = 3) -> dict[str, Any]:
    query = str(row["query"])
    expected_source = str(row["expected_source_type"])
    csv_expected_ids = parse_doc_ids(row.get("expected_doc_ids", ""))
    expected_ids = expected_ids_from_row(row)
    router = QueryRouter(mode="heuristic")

    with start_span("notebook.router_retrieval.row", dataset=str(row["dataset"]), query_length=len(query)):
        predicted_source = (await router.route(query)).value
        route_correct = predicted_source == expected_source

        async def retrieve_for(source_value: str) -> tuple[list[str], list[str]]:
            if not is_local_source(source_value):
                return [], []
            retriever = ChromaRetriever(chroma_path=str(settings.chroma_path), top_k=top_k)
            docs = await retriever.retrieve(SourceType(source_value), query)
            return [doc.doc_id for doc in docs], [doc.text for doc in docs]

        expected_retrieved_ids, expected_docs = await retrieve_for(expected_source)
        predicted_retrieved_ids, predicted_docs = await retrieve_for(predicted_source)

        expected_score = score_retrieval(expected_retrieved_ids, expected_ids)
        predicted_score = score_retrieval(predicted_retrieved_ids, expected_ids)
        has_gold = bool(expected_ids)

        return {
            "dataset": row["dataset"],
            "query": query,
            "expected_source_type": expected_source,
            "predicted_source_type": predicted_source,
            "route_correct": route_correct,
            "csv_expected_doc_ids": "|".join(csv_expected_ids),
            "expected_doc_ids": "|".join(expected_ids),
            "expected_source_retrieved_doc_ids": "|".join(expected_retrieved_ids),
            "predicted_source_retrieved_doc_ids": "|".join(predicted_retrieved_ids),
            "expected_source_hit_at_k": expected_score.hit_at_k if has_gold else None,
            "predicted_source_hit_at_k": predicted_score.hit_at_k if has_gold else None,
            "expected_source_recall_at_k": expected_score.recall_at_k if has_gold else None,
            "predicted_source_recall_at_k": predicted_score.recall_at_k if has_gold else None,
            "route_induced_retrieval_failure": bool(has_gold and expected_score.hit_at_k == 1 and predicted_score.hit_at_k == 0),
            "top_predicted_context_preview": preview_text(predicted_docs[0]) if predicted_docs else None,
            "metric_status": "scored" if has_gold else ("qualitative_only_no_gold_doc_ids" if is_local_source(expected_source) else "skipped_web_source"),
        }


async def evaluate_router_retrieval(df: pd.DataFrame, top_k: int = 3) -> pd.DataFrame:
    rows = await asyncio.gather(*(route_and_retrieve_one(row, top_k) for _, row in df.iterrows()))
    return pd.DataFrame(rows)


## Measure Cascade On Clear Examples


In [ ]:
base_cascade_results = await evaluate_router_retrieval(base_df, top_k=3)
base_cascade_results.head()


In [ ]:
base_scored = base_cascade_results[base_cascade_results["metric_status"] == "scored"]
base_summary = pd.DataFrame([
    {
        "dataset": "base",
        "examples": len(base_cascade_results),
        "router_accuracy": float(base_cascade_results["route_correct"].mean()),
        "expected_source_hit_at_3": float(base_scored["expected_source_hit_at_k"].mean()),
        "predicted_source_hit_at_3": float(base_scored["predicted_source_hit_at_k"].mean()),
        "route_induced_retrieval_failures": int(base_scored["route_induced_retrieval_failure"].sum()),
    }
])
base_summary


## Inspect Cascade Behavior On Hard Examples


In [ ]:
challenge_cascade_results = await evaluate_router_retrieval(challenge_df, top_k=3)
challenge_cascade_results.head()


In [ ]:
challenge_summary = pd.DataFrame([
    {
        "dataset": "challenging",
        "examples": len(challenge_cascade_results),
        "router_accuracy": float(challenge_cascade_results["route_correct"].mean()),
        "qualitative_rows": int((challenge_cascade_results["metric_status"] == "qualitative_only_no_gold_doc_ids").sum()),
        "skipped_web_rows": int((challenge_cascade_results["metric_status"] == "skipped_web_source").sum()),
    }
])
challenge_summary


In [ ]:
challenge_cascade_results.loc[
    ~challenge_cascade_results["route_correct"],
    ["query", "expected_source_type", "predicted_source_type", "metric_status", "top_predicted_context_preview"],
]


## Pay Attention To

- A route can be correct while retrieval still misses the gold document; keep these failures separate.
- Expected-source retrieval is a useful control because it shows what retrieval could do if routing were perfect.
- Predicted-source retrieval shows the real cascade effect from the router into retrieval.
- The most useful rows are often the disagreements between expected-source and predicted-source retrieval.


## Optional Advanced Path: LLM Router Evaluation

This section is disabled by default because it makes API calls. Set `RUN_LLM_ROUTER = True` to compare the LLM router against both the base and challenging router datasets. It reads `OPENAI_API_KEY` from the active environment only; it does not load `.env`.


In [ ]:
RUN_LLM_ROUTER = True
LLM_ROUTER_MODEL = os.getenv("OPENAI_ROUTER_MODEL", "gpt-5-nano")
LLM_ROUTER_TIMEOUT = float(os.getenv("OPENAI_TIMEOUT", "60"))

llm_router_results = pd.DataFrame()
llm_router_summary = pd.DataFrame()


def load_router_eval_dataset(path: Path, dataset_name: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    if "Query" in df.columns:
        df = df.rename(columns={"Query": "query", "Expected_Source_Type": "expected_source_type"})
    df = df.copy()
    df["dataset"] = dataset_name
    df["expected_source_type"] = df["expected_source_type"].astype(str)
    return df


async def evaluate_llm_router_dataset(router: QueryRouter, df: pd.DataFrame) -> pd.DataFrame:
    async def route_row(query: str) -> str:
        return (await router.route(query)).value

    predictions = await asyncio.gather(*(route_row(str(query)) for query in df["query"].tolist()))
    results = df.copy()
    results["router_mode"] = "llm"
    results["predicted_source_type"] = predictions
    results["route_correct"] = results["expected_source_type"] == results["predicted_source_type"]
    keep_columns = [
        "dataset",
        "router_mode",
        "query",
        "expected_source_type",
        "predicted_source_type",
        "route_correct",
        "category",
        "rationale",
    ]
    return results[[column for column in keep_columns if column in results.columns]]


def summarize_llm_router_results(results: pd.DataFrame) -> pd.DataFrame:
    if results.empty:
        return pd.DataFrame()
    return results.groupby(["dataset", "router_mode"], dropna=False).agg(
        examples=("query", "count"),
        accuracy=("route_correct", "mean"),
        failures=("route_correct", lambda values: int((~values).sum())),
    ).reset_index()


if RUN_LLM_ROUTER:
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        raise RuntimeError("OPENAI_API_KEY must be set in the active environment for LLM router evaluation")

    llm_router = QueryRouter(
        generator=OpenAITextGenerator(api_key=api_key, model=LLM_ROUTER_MODEL, timeout=LLM_ROUTER_TIMEOUT),
        mode="llm",
    )
    llm_router_base_df = load_router_eval_dataset(PROJECT_ROOT / "datasets/evaluation_dataset.csv", "base")
    llm_router_challenge_df = load_router_eval_dataset(PROJECT_ROOT / "datasets/challenging_router_evaluation_dataset.csv", "challenging")
    llm_router_results = pd.concat(
        [
            await evaluate_llm_router_dataset(llm_router, llm_router_base_df),
            await evaluate_llm_router_dataset(llm_router, llm_router_challenge_df),
        ],
        ignore_index=True,
    )
    llm_router_summary = summarize_llm_router_results(llm_router_results)

llm_router_summary


In [ ]:
if not llm_router_results.empty:
    display(llm_router_summary)
    display(pd.crosstab(
        [llm_router_results["dataset"], llm_router_results["expected_source_type"]],
        llm_router_results["predicted_source_type"],
        dropna=False,
    ))
    display(llm_router_results.loc[~llm_router_results["route_correct"]])


## Export Cascade Artifacts


In [ ]:
cascade_results = pd.concat([base_cascade_results, challenge_cascade_results], ignore_index=True)
cascade_summary = pd.concat([base_summary, challenge_summary], ignore_index=True)

results_path = PROJECT_ROOT / "router_retrieval_cascade_results.csv"
summary_path = PROJECT_ROOT / "router_retrieval_cascade_summary.csv"
cascade_results.to_csv(results_path, index=False)
cascade_summary.to_csv(summary_path, index=False)

results_path, summary_path
